In [1]:
# Default party context demo - no explicit 'with pp.Party()' needed!
import poolparty as pp

# Create pools directly - works immediately after import
pool = pp.from_seqs(['AAA', 'TTT', 'GGG'], mode='sequential').named('seq')
df = pool.generate_seqs(num_seqs=3)
df

,seq,seq.seq,seq.state,seq.op.state,seq.op.key.seq_name,seq.op.key.seq_index
0,AAA,AAA,0,0,seq_0,0
1,TTT,TTT,1,1,seq_1,1
2,GGG,GGG,2,2,seq_2,2


In [2]:
# More complex operations also work without explicit context
mutants = pp.mutagenize('ACGT', num_mutations=1, mode='sequential').named('mutant')
df = mutants.generate_seqs(num_seqs=5)
df

,seq,mutant.seq,op[1]:from_seq.key.seq,pool[1].seq,mutant.state,pool[1].state,mutant.op.state,op[1]:from_seq.state,mutant.op.key.positions,mutant.op.key.wt_chars,mutant.op.key.mut_chars,op[1]:from_seq.key.seq
0,CCGT,CCGT,ACGT,ACGT,0,0,0,0,"(0,)","(A,)","(C,)",ACGT
1,GCGT,GCGT,ACGT,ACGT,1,0,1,0,"(0,)","(A,)","(G,)",ACGT
2,TCGT,TCGT,ACGT,ACGT,2,0,2,0,"(0,)","(A,)","(T,)",ACGT
3,AAGT,AAGT,ACGT,ACGT,3,0,3,0,"(1,)","(C,)","(A,)",ACGT
4,AGGT,AGGT,ACGT,ACGT,4,0,4,0,"(1,)","(C,)","(G,)",ACGT


In [3]:
# Join pools and use markers without context manager
left = pp.from_seqs(['AAA'])
right = pp.from_seqs(['TTT'])
oligo = pp.join([left, '---', right]).named('oligo')
df = oligo.generate_seqs(num_seqs=1)
df

,seq,oligo.seq,pool[3].seq,op[5]:from_seq.key.seq,pool[5].seq,pool[4].seq,oligo.state,pool[3].state,pool[5].state,pool[4].state,oligo.op.state,op[3]:from_seqs.state,op[5]:from_seq.state,op[4]:from_seqs.state,op[3]:from_seqs.key.seq_name,op[3]:from_seqs.key.seq_index,op[5]:from_seq.key.seq,op[4]:from_seqs.key.seq_name,op[4]:from_seqs.key.seq_index
0,AAA---TTT,AAA---TTT,AAA,---,---,TTT,0,0,0,0,0,0,0,0,seq_0,0,---,seq_0,0


In [4]:
# reset_default_party() clears state and optionally changes alphabet
pp.reset_default_party(alphabet='rna')
rna_pool = pp.from_seqs(['AAA', 'UUU'], mode='sequential').named('rna_seq')
df = rna_pool.generate_seqs(num_seqs=2)
df

,seq,rna_seq.seq,rna_seq.state,rna_seq.op.state,rna_seq.op.key.seq_name,rna_seq.op.key.seq_index
0,AAA,AAA,0,0,seq_0,0
1,UUU,UUU,1,1,seq_1,1


In [5]:
# Explicit Party context still works - temporarily replaces default
pp.reset_default_party()  # back to DNA
with pp.Party(alphabet='protein') as party:
    protein_pool = pp.get_kmers(length=2, mode='sequential').named('dipeptide')
    df = protein_pool.generate_seqs(num_seqs=5)
    print(f"Inside explicit context: {pp.get_active_party().alphabet.chars[:5]}...")
print(f"After exit: {pp.get_active_party().alphabet.chars[:4]}")
df

Inside explicit context: ['A', 'C', 'D', 'E', 'F']...
After exit: ['A', 'C', 'G', 'T']


,seq,dipeptide.seq,dipeptide.state,dipeptide.op.state,dipeptide.op.key.kmer_index
0,AA,AA,0,0,0
1,AC,AC,1,1,1
2,AD,AD,2,2,2
3,AE,AE,3,3,3
4,AF,AF,4,4,4


In [6]:
# Nested contexts now work (they stack)
pp.reset_default_party()
with pp.Party(alphabet='dna') as outer:
    print(f"Outer party active, alphabet: {outer.alphabet.chars}")
    with pp.Party(alphabet='rna') as inner:
        print(f"Inner party active, alphabet: {inner.alphabet.chars}")
    print(f"Back to outer, alphabet: {pp.get_active_party().alphabet.chars}")
print(f"Back to default, alphabet: {pp.get_active_party().alphabet.chars}")

Outer party active, alphabet: ['A', 'C', 'G', 'T']
Inner party active, alphabet: ['A', 'C', 'G', 'U']
Back to outer, alphabet: ['A', 'C', 'G', 'T']
Back to default, alphabet: ['A', 'C', 'G', 'T']
